In [2]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [3]:
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

2024-05-13 14:09:27.089147: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-05-13 14:09:27.095359: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-05-13 14:09:27.170976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-13 14:09:28.592263: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Params

In [3]:
meta_neuron_array = np.array([1,2,4,8,93])
N_training_repeats = 10
cg_params = {
    "order": 1,
}
critic_params = {
    "hidden_dim": 32,
    "embed_dim": 30,
    "layers": 2,
    "activation": "tanh",
}
opt_params = {"track_quantities": False, "quiet": True, "iterations": 8}
mi_measurement_params = {'quiet':True, 'batch_size':500, 'N_batches':500}

V_width = 1
V_E_buffer = 0
E_width = 1

train_fraction = 0.5
samp_keep = 1

# Put variable names here for parameters we want saved.
save_vars = {
    'mi_measurement_params',
    'meta_neuron_array', 
    'N_training_repeats', 
    'cg_params', 
    'critic_params', 
    'opt_params', 
    'V_width', 
    'V_E_buffer', 
    'E_width', 
    'train_fraction', 
    'samp_keep', 
    }

In [4]:
meta_neuron_array

array([ 1,  2,  4,  8, 93])

## Prepare data

In [5]:
data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
D = ds.DataNatScenes(data_path, mov_ind=3)
mov_ind = D.mov_ind
mov_name = D.movname
save_vars.add('mov_ind')

# data_path = os.path.join(project_folder, 'data', 'bar_stim_data')
# D = ds.DataBar(data_path)
# mov_name = 'bar'

save_vars.add('mov_name')
print(f'current stimulus is {mov_name}')

current stimulus is fish


In [6]:
D.reset()
D.trim(20)  # remove first 20 frames from each trial
V, E = D.create_V_and_E_datasets(V_width, V_E_buffer, E_width, samp_keep=samp_keep)

## Run analysis

In [7]:
experiment_dims = (len(meta_neuron_array), N_training_repeats)
mi_averages_test = np.zeros(experiment_dims)
mi_averages_train = np.zeros(experiment_dims)
mi_stdevs_train = np.zeros(experiment_dims)
mi_stdevs_test = np.zeros(experiment_dims)

train_points = np.zeros(experiment_dims, dtype=object)
test_points = np.zeros(experiment_dims, dtype=object)


for i, N_meta_neurons in tqdm(enumerate(meta_neuron_array)):
    for training_repeat in range(N_training_repeats):
        coarse_grainer = mds.PolynomialLayer(output_dim=N_meta_neurons, **cg_params)
        critic_function = mds.SeparableCritic(**critic_params)

        # make training and test subsets
        V.reset()
        E.reset()
        current_train_points, current_test_points = tools.generate_train_test_indices(
            V.P, train_fraction
        )
        V.subset_points(current_train_points)
        E.subset_points(current_train_points)
        train_points[i, training_repeat] = current_train_points
        test_points[i, training_repeat] = current_test_points

        # Train current
        als.IB(
            X=V.X,
            Y=E.X,
            coarse_grainer=coarse_grainer,
            critic_function=critic_function,
            **opt_params
        )

        # Measure InfoNCE for train
        T_train = coarse_grainer(V.X.T).numpy().T
        mi_train = als.MI_NCE(
            X=T_train, Y=E.X, critic_function=critic_function, **mi_measurement_params
        )
        mi_averages_train[i, training_repeat] = mi_train["mi"]
        mi_stdevs_train[i, training_repeat] = mi_train["mi_stdev"]

        # Measure InfoNCE for test
        V.reset()
        E.reset()
        V.subset_points(current_test_points)
        E.subset_points(current_test_points)
        T_test = coarse_grainer(V.X.T).numpy().T
        mi_test = als.MI_NCE(
            X=T_test, Y=E.X, critic_function=critic_function, **mi_measurement_params
        )
        mi_averages_test[i, training_repeat] = mi_test["mi"]
        mi_stdevs_test[i, training_repeat] = mi_test["mi_stdev"]

0it [00:00, ?it/s]2024-05-11 18:25:32.922690: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-05-11 18:25:32.923675: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


5it [10:11, 122.34s/it]


## Save

In [8]:
# select variables
coarse_grainer_type = type(coarse_grainer)
critic_type = type(critic_function)
save_vars.update({'mi_averages_train', 'mi_averages_test', 'mi_stdevs_train', 'mi_stdevs_test', 'coarse_grainer_type', 'critic_type'})
local_dict = locals()
output_data = {k: local_dict[k] for k in save_vars}

In [9]:
# save things
timestamp = strftime("%Y.%m.%d %Hh %Mm %Ss.dat",gmtime())
# full_file_path = os.path.join(save_directory, strftime("%Y.%m.%d %Hh %Mm %Ss",gmtime())+'.dat')
stim = mov_name
# full_file_path = os.path.join(save_directory, stim, 'all neurons small K.dat')
full_file_path = os.path.join(save_directory, stim, 'all neurons small K large iter.dat')
print(f"data will be saved to:\n{full_file_path}")

data will be saved to:
/home/adam/Documents/projects/retina-past-future-ib/analyses/IB curves/saved/fish/all neurons small K large iter.dat


In [10]:
os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
with open(full_file_path, 'wb+') as f:
            pickle.dump(output_data, f)